In [ ]:
from os import listdir
import json
from tqdm import tqdm
import multiprocessing as mp
from os.path import exists
from MultiProcessSocket import MultiProcessSocket

In [ ]:
paths=["../web crawler/files/ckip-topics/nchu-topic/",
       "../web crawler/files/ckip-topics/nkust-topic/",
       "../web crawler/files/ckip-topics/ntnu-topic/",
       "../web crawler/files/ckip-topics/nuk-topic/",
      ]
skiptags=['A', 'Caa', 'Cab', 'Cba', 'D', 'Da', 'Dfa', 'Dfb', 'Di', 'Dk', 'DM', 'I',  'Nb', 'Nc', 'Ncd', 'Nep', 'Neqb', 'Nes', 'Nf', 'Ng', 'Nh', 'Nv', 'P', 'T', 'VA', 'VAC', 'VB', 'VC', 'VCL', 'VD', 'VF', 'VE', 'VG', 'VH', 'VHC', 'VI', 'VJ', 'VK', 'VL', 'V_2', 'DE', 'Cbb', 'Nd', 'Neu', 'Neqa', 'SHI', 'FW', 'COLONCATEGORY', 'COMMACATEGORY', 'DASHCATEGORY', 'DOTCATEGORY', 'ETCCATEGORY', 'EXCLAMATIONCATEGORY', 'PARENTHESISCATEGORY', 'PAUSECATEGORY', 'PERIODCATEGORY', 'QUESTIONCATEGORY', 'SEMICOLONCATEGORY', 'SPCHANGECATEGORY', 'WHITESPACE']
#'Na', 
topics=[]
for path in paths:
    for file in listdir(path):
        topics.append(path + file) 

In [ ]:
def add_word(dictionary,words):
    for w in words:
        if w in dictionary['word_indexs']:            
            index = dictionary['word_indexs'][w]
            dictionary['word_set'][index][1] += 1
        else:
            dictionary['word_indexs'][w] = dictionary['word_counter']
            dictionary['word_set'][dictionary['word_counter']][0] = w
            dictionary['word_set'][dictionary['word_counter']][1] = 1
            dictionary['word_counter'] += 1

In [ ]:
#findtag = ["張貼","活動組","活動","聯絡","建工","方式","教師","期間","附件","小姐"]
def worker(manager, path):
    with open(path, mode="r") as f: 
        jsonn = json.loads(f.read())
        words=[]
        try:
            for info in jsonn["content"]:
                for i in range(len(info)):
                    add = True
                    if len(info[i]['word']) < 2:
                        add = False
                    for skiptag in skiptags:
                        if info[i]['tag'] == skiptag:
                            add = False
                            break
                    if add:
                        words.append(info[i]['word'])
                        '''
                        if info[i]['word'] in findtag:
                            print(info[i]['word'] + " : " + info[i]['tag'])
                        '''

            manager.call(words)
        except Exception as e:
            print(e)
            print(path)

In [ ]:
def main(): 
    # "mp.cpu_count() + 2" Common practice to ensure that the pool will be saturated.
    pool = mp.Pool(mp.cpu_count()*2)    
    manager = MultiProcessSocket(
                add_word,
                {
                    'word_indexs':{},
                    'word_set':[["",0] for i in range(300000)],
                    'word_counter':0    
                }
              )
    manager.start()
    #pool.apply_async(s.start, (add_word,))
    progress = tqdm(total=len(topics))
    #fire off workers
    jobs = []
    for i in topics:
        job = pool.apply_async(worker, (manager,i))
        jobs.append(job)

    # collect results from the workers through the pool result queue
    for job in jobs:
        job.wait() 
        #job.get()
        progress.update(1)  
        
    #print("main:"+str(id(d.word_set))+"\n",end="")
    with open("files/dictionary.txt","w",encoding="utf-8") as f:
        d = manager.get_data()
        dictionary = d['word_set'][:d['word_counter']]
        dictionary.sort(key=lambda x: x[1], reverse=True)        
        f.write(json.dumps(dictionary, indent=4, ensure_ascii=False))
        print("finish")
    manager.close()
    pool.close()
    pool.join()

In [ ]:
if __name__ == "__main__":
    main()